In [ ]:
# Enhanced Swin Transformer for Skin Cancer Classification
# With ViT-specific XAI: Attention Rollout, Attention Flow, and Transformer Interpretability
# OPTIMIZED FOR MEMORY EFFICIENCY - Kaggle Environment

import os
import gc
import pandas as pd
import numpy as np
from PIL import Image
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import seaborn as sns
from torchvision.models import swin_v2_b, Swin_V2_B_Weights
import cv2
import warnings
warnings.filterwarnings('ignore')

# Memory optimization settings
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

# Create output directory
output_dir = "visualization_outputs"
os.makedirs(output_dir, exist_ok=True)

# Memory management utilities
def clear_memory():
    """Clear GPU and CPU memory"""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    gc.collect()

def force_clear_memory():
    """Force clear all cached memory"""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        torch.cuda.synchronize()
    gc.collect()

# Enhanced device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.cuda.set_per_process_memory_fraction(0.9)
    print(f"Using GPU: {torch.cuda.get_device_name()}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

# ✅ STEP 1: Enhanced Data Loading with New Path Structure
def load_and_preprocess_data(base_path):
    """Load and preprocess the HAM10000 dataset from new structure"""

    # Define class names from directory structure
    class_names = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']

    # Create lists for data
    image_ids = []
    paths = []
    dx_labels = []
    splits = []

    train_dir = os.path.join(base_path, 'base_dir', 'train_dir')
    val_dir = os.path.join(base_path, 'base_dir', 'val_dir')

    print("Loading training data...")
    for class_name in class_names:
        class_path = os.path.join(train_dir, class_name)
        if os.path.exists(class_path):
            for img_file in os.listdir(class_path):
                if img_file.endswith(('.jpg', '.jpeg', '.png')):
                    image_ids.append(os.path.splitext(img_file)[0])
                    paths.append(os.path.join(class_path, img_file))
                    dx_labels.append(class_name)
                    splits.append('train')

    print("Loading validation data...")
    for class_name in class_names:
        class_path = os.path.join(val_dir, class_name)
        if os.path.exists(class_path):
            for img_file in os.listdir(class_path):
                if img_file.endswith(('.jpg', '.jpeg', '.png')):
                    image_ids.append(os.path.splitext(img_file)[0])
                    paths.append(os.path.join(class_path, img_file))
                    dx_labels.append(class_name)
                    splits.append('val')

    # Create dataframe with explicit column assignment
    df = pd.DataFrame({
        'image_id': image_ids,
        'path': paths,
        'dx': dx_labels,
        'split': splits
    })

    # Label encoding
    label_map = {label: idx for idx, label in enumerate(sorted(class_names))}
    df['label'] = df['dx'].map(label_map)

    # Print statistics
    print("\nDataset Statistics:")
    print(f"Total images: {len(df)}")
    print(f"Training images: {len(df[df['split']=='train'])}")
    print(f"Validation images: {len(df[df['split']=='val'])}")

    print("\nClass Distribution:")
    for split in ['train', 'val']:
        print(f"\n{split.upper()}:")
        class_counts = df[df['split']==split]['dx'].value_counts()
        for class_name, count in class_counts.items():
            print(f"  {class_name}: {count}")

    return df, label_map

# ✅ STEP 2: Dataset Class
class HAM10000Dataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.loc[idx, 'path']
        label = self.df.loc[idx, 'label']

        try:
            with Image.open(img_path) as img:
                image = img.convert("RGB").copy()
        except Exception as e:
            print(f"Error loading image {img_path}: {e}")
            image = Image.new('RGB', (224, 224), color='black')

        if self.transform:
            image = self.transform(image)

        return image, label, img_path

# ✅ STEP 3: Transforms
def get_transforms(img_size=224, mode='train'):
    if mode == 'train':
        return transforms.Compose([
            transforms.Resize((img_size + 16, img_size + 16)),
            transforms.RandomCrop((img_size, img_size)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.3),
            transforms.RandomRotation(degrees=15),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            transforms.RandomErasing(p=0.1, scale=(0.02, 0.2))
        ])
    else:
        return transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

# ✅ STEP 4: Enhanced Swin Transformer with Attention Extraction
class EnhancedSwinTransformer(nn.Module):
    def __init__(self, num_classes, dropout_rate=0.2, pretrained=True):
        super(EnhancedSwinTransformer, self).__init__()

        if pretrained:
            weights = Swin_V2_B_Weights.IMAGENET1K_V1
            self.backbone = swin_v2_b(weights=weights)
        else:
            self.backbone = swin_v2_b(weights=None)

        # Store attention weights
        self.attention_maps = []
        self.attention_hooks = []

        num_features = self.backbone.head.in_features
        self.backbone.head = nn.Identity()

        self.classifier = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(num_features, num_features // 4),
            nn.ReLU(),
            nn.BatchNorm1d(num_features // 4),
            nn.Dropout(dropout_rate / 2),
            nn.Linear(num_features // 4, num_classes)
        )

        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.constant_(m.bias, 0)

    def register_attention_hooks(self):
        """Register hooks to capture attention weights from Swin blocks"""
        self.attention_maps = []
        self.attention_hooks = []

        def get_attention(name):
            def hook(module, input, output):
                # Store attention weights
                if hasattr(module, 'attn') and hasattr(module.attn, 'attention_weights'):
                    self.attention_maps.append(module.attn.attention_weights.detach())
            return hook

        # Register hooks on Swin Transformer blocks
        for name, module in self.backbone.named_modules():
            if 'blocks' in name and isinstance(module, torch.nn.Module):
                handle = module.register_forward_hook(get_attention(name))
                self.attention_hooks.append(handle)

    def remove_attention_hooks(self):
        """Remove attention hooks"""
        for handle in self.attention_hooks:
            handle.remove()
        self.attention_hooks = []
        self.attention_maps = []

    def forward(self, x):
        # Handle edge case of single sample batch during training
        if self.training and x.size(0) == 1:
            # Add a dummy batch dimension then remove it
            x_expanded = x.repeat(2, 1, 1, 1)
            features = self.backbone(x_expanded)
            output = self.classifier(features)
            return output[:1]  # Return only the first sample

        features = self.backbone(x)
        return self.classifier(features)

# ✅ STEP 5: ViT-Specific XAI Tools

class AttentionRollout:
    """Attention Rollout for Vision Transformers"""

    def __init__(self, model, attention_layer_name='attn', head_fusion='mean'):
        self.model = model
        self.attention_layer_name = attention_layer_name
        self.head_fusion = head_fusion
        self.attention_maps = []

    def get_attention_maps(self, input_tensor):
        """Extract attention maps from model"""
        self.model.eval()
        self.attention_maps = []

        # Register hooks to capture attention
        hooks = []

        def get_attention_hook(name):
            def hook(module, input, output):
                if hasattr(module, 'attn_drop'):
                    # This is an attention layer
                    # Try to get attention weights if available
                    if len(input) > 0 and isinstance(input[0], torch.Tensor):
                        self.attention_maps.append(input[0].detach().cpu())
            return hook

        # Register hooks on attention layers
        for name, module in self.model.backbone.named_modules():
            if self.attention_layer_name in name:
                handle = module.register_forward_hook(get_attention_hook(name))
                hooks.append(handle)

        # Forward pass
        with torch.no_grad():
            _ = self.model(input_tensor)

        # Remove hooks
        for handle in hooks:
            handle.remove()

        return self.attention_maps

    def compute_rollout(self, input_tensor, start_layer=0):
        """Compute attention rollout"""
        attention_maps = self.get_attention_maps(input_tensor)

        if len(attention_maps) == 0:
            # If no attention maps extracted, create visualization from features
            return self.create_feature_based_attention(input_tensor)

        # Process attention maps
        rollout = None
        for i, attn in enumerate(attention_maps[start_layer:]):
            if attn.dim() == 4:  # [batch, heads, seq, seq]
                # Average over heads
                attn = attn.mean(dim=1)

            if rollout is None:
                rollout = attn
            else:
                rollout = torch.matmul(attn, rollout)

        return rollout

    def create_feature_based_attention(self, input_tensor):
        """Create attention-like visualization from features"""
        self.model.eval()

        # Get intermediate features
        features_dict = {}

        def get_features_hook(name):
            def hook(module, input, output):
                features_dict[name] = output.detach()
            return hook

        hooks = []
        # Register hooks on key layers
        for name, module in self.model.backbone.named_modules():
            if 'norm' in name or 'blocks.3' in name:
                handle = module.register_forward_hook(get_features_hook(name))
                hooks.append(handle)

        with torch.no_grad():
            _ = self.model(input_tensor)

        for handle in hooks:
            handle.remove()

        # Use the last feature map
        if features_dict:
            last_features = list(features_dict.values())[-1]

            # Compute attention-like map from features
            if last_features.dim() == 3:  # [batch, seq_len, features]
                attn_weights = torch.softmax(last_features.mean(dim=-1), dim=-1)
            else:
                attn_weights = torch.ones(1, 49)  # Default 7x7 grid

            return attn_weights.cpu()

        return torch.ones(1, 49).cpu()

    def visualize(self, input_tensor, original_image, save_path=None):
        """Visualize attention rollout"""
        rollout = self.compute_rollout(input_tensor)

        # Convert to 2D attention map
        if rollout.dim() == 2:
            attn_map = rollout[0].numpy()
        else:
            attn_map = rollout.mean(dim=0).numpy()

        # Reshape to 2D if needed
        map_size = int(np.sqrt(attn_map.shape[-1]))
        if attn_map.shape[-1] == map_size * map_size:
            attn_map = attn_map.reshape(map_size, map_size)
        else:
            # Use first dimension
            attn_map = attn_map[:map_size*map_size].reshape(map_size, map_size)

        # Resize to input size
        attn_map = cv2.resize(attn_map, (224, 224))
        attn_map = (attn_map - attn_map.min()) / (attn_map.max() - attn_map.min() + 1e-8)

        # Visualize
        fig, axes = plt.subplots(1, 3, figsize=(12, 4))

        axes[0].imshow(original_image)
        axes[0].set_title('Original Image')
        axes[0].axis('off')

        axes[1].imshow(attn_map, cmap='jet')
        axes[1].set_title('Attention Rollout')
        axes[1].axis('off')

        axes[2].imshow(original_image)
        axes[2].imshow(attn_map, cmap='jet', alpha=0.5)
        axes[2].set_title('Attention Overlay')
        axes[2].axis('off')

        plt.tight_layout()

        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
            print(f"Attention Rollout saved to {save_path}")

        plt.show()
        plt.close()
        clear_memory()
        return fig

class TransformerAttributionAnalysis:
    """Multi-layer attention analysis for transformers"""

    def __init__(self, model):
        self.model = model

    def extract_layer_features(self, input_tensor):
        """Extract features from different layers"""
        self.model.eval()
        layer_features = {}

        def get_features(name):
            def hook(module, input, output):
                layer_features[name] = output.detach().cpu()
            return hook

        hooks = []
        layer_names = []

        # Register hooks on key layers
        for name, module in self.model.backbone.named_modules():
            if 'features' in name and ('0' in name or '3' in name or '5' in name or '7' in name):
                handle = module.register_forward_hook(get_features(name))
                hooks.append(handle)
                layer_names.append(name)

        with torch.no_grad():
            _ = self.model(input_tensor)

        for handle in hooks:
            handle.remove()

        return layer_features, layer_names

    def visualize_layer_attention(self, input_tensor, original_image, save_path=None):
        """Visualize attention across layers"""
        layer_features, layer_names = self.extract_layer_features(input_tensor)

        if len(layer_features) == 0:
            print("No layer features extracted")
            return None

        num_layers = min(4, len(layer_features))
        fig, axes = plt.subplots(2, num_layers, figsize=(4*num_layers, 8))

        if num_layers == 1:
            axes = axes.reshape(2, 1)

        for idx, (layer_name, features) in enumerate(list(layer_features.items())[:num_layers]):
            # Process features
            if features.dim() == 4:  # [B, C, H, W]
                feat_map = features[0].mean(dim=0).numpy()
            elif features.dim() == 3:  # [B, H*W, C]
                feat = features[0].mean(dim=-1).numpy()
                size = int(np.sqrt(len(feat)))
                if len(feat) == size * size:
                    feat_map = feat.reshape(size, size)
                else:
                    feat_map = np.zeros((7, 7))
            else:
                feat_map = np.zeros((7, 7))

            # Normalize
            feat_map = (feat_map - feat_map.min()) / (feat_map.max() - feat_map.min() + 1e-8)
            feat_map_resized = cv2.resize(feat_map, (224, 224))

            # Plot feature map
            axes[0, idx].imshow(feat_map_resized, cmap='viridis')
            axes[0, idx].set_title(f'Layer {idx+1}')
            axes[0, idx].axis('off')

            # Plot overlay
            axes[1, idx].imshow(original_image)
            axes[1, idx].imshow(feat_map_resized, cmap='jet', alpha=0.4)
            axes[1, idx].set_title(f'Layer {idx+1} Overlay')
            axes[1, idx].axis('off')

        plt.suptitle('Multi-Layer Attention Analysis', fontsize=14)
        plt.tight_layout()

        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
            print(f"Layer attention analysis saved to {save_path}")

        plt.show()
        plt.close()
        clear_memory()
        return fig

class RelevanceMapGenerator:
    """Generate relevance maps using gradient-based attribution"""

    def __init__(self, model):
        self.model = model

    def generate_relevance_map(self, input_tensor, target_class=None):
        """Generate relevance map using gradients"""
        self.model.eval()
        input_tensor.requires_grad = True

        # Forward pass
        output = self.model(input_tensor)

        if target_class is None:
            target_class = output.argmax(dim=1)

        # Backward pass
        self.model.zero_grad()
        output[0, target_class].backward()

        # Get gradients
        gradients = input_tensor.grad.data

        # Compute relevance
        relevance = (gradients * input_tensor).sum(dim=1, keepdim=True)
        relevance = torch.abs(relevance)

        return relevance.detach().cpu()

    def visualize(self, input_tensor, original_image, save_path=None):
        """Visualize relevance map"""
        relevance = self.generate_relevance_map(input_tensor)

        # Process relevance map
        rel_map = relevance[0, 0].numpy()
        rel_map = (rel_map - rel_map.min()) / (rel_map.max() - rel_map.min() + 1e-8)

        # Resize
        rel_map = cv2.resize(rel_map, (224, 224))

        # Visualize
        fig, axes = plt.subplots(1, 3, figsize=(12, 4))

        axes[0].imshow(original_image)
        axes[0].set_title('Original Image')
        axes[0].axis('off')

        axes[1].imshow(rel_map, cmap='hot')
        axes[1].set_title('Relevance Map')
        axes[1].axis('off')

        axes[2].imshow(original_image)
        axes[2].imshow(rel_map, cmap='hot', alpha=0.5)
        axes[2].set_title('Relevance Overlay')
        axes[2].axis('off')

        plt.tight_layout()

        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
            print(f"Relevance map saved to {save_path}")

        plt.show()
        plt.close()
        clear_memory()
        return fig

# ✅ STEP 6: Enhanced Trainer with ViT-XAI
class EnhancedTrainer:
    def __init__(self, model, train_loader, val_loader, num_classes, label_map, device):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.num_classes = num_classes
        self.label_map = label_map
        self.device = device

        self.setup_loss_function()
        self.setup_optimizer()

        self.train_losses = []
        self.val_losses = []
        self.train_accuracies = []
        self.val_accuracies = []
        self.best_val_acc = 0.0
        self.best_model_state = None

    def setup_loss_function(self):
        # Collect labels to compute class weights
        train_labels = []
        for batch_idx, (_, labels, _) in enumerate(self.train_loader):
            train_labels.extend(labels.numpy())
            if batch_idx > 100:  # Enough for a good estimate; keep it fast
                break

        # Step 1: Compute initial balanced weights
        class_weights_balanced = compute_class_weight(
            'balanced',
            classes=np.unique(train_labels),
            y=train_labels
        )
        class_weights_balanced = torch.FloatTensor(class_weights_balanced).to(self.device)

        # Step 2: Apply targeted boost for difficult classes (mel, bkl)
        # Based on the provided confusion matrix, mel (0.890) and bkl (0.918) are the weakest.
        # We'll boost them relative to the average of the top 3 performing classes (akiec: 0.972, bcc: 0.996, df: 0.983, nv: 0.970, vasc: 1.000).
        # Let's use a dynamic factor: Boost = (Average_Top_Accuracy / Class_Accuracy) ^ 1.5
        # This gives a significant but not overwhelming boost.
        avg_top_accuracy = (0.972 + 0.996 + 0.983 + 0.970 + 1.000) / 5  # ~0.984
        mel_accuracy = 0.890
        bkl_accuracy = 0.918

        mel_boost_factor = (avg_top_accuracy / mel_accuracy) ** 1.5  # ~1.35
        bkl_boost_factor = (avg_top_accuracy / bkl_accuracy) ** 1.5  # ~1.20

        # Get indices for mel and bkl
        mel_class_index = list(self.label_map.keys()).index('mel')  # 'mel' is at index 4
        bkl_class_index = list(self.label_map.keys()).index('bkl')  # 'bkl' is at index 2

        # Apply boosts
        class_weights_balanced[mel_class_index] *= mel_boost_factor
        class_weights_balanced[bkl_class_index] *= bkl_boost_factor

        print(f"Class Weights after enhancement:")
        for i, class_name in enumerate(self.label_map.keys()):
            print(f"  {class_name}: {class_weights_balanced[i]:.4f}")

        self.criterion = nn.CrossEntropyLoss(weight=class_weights_balanced)
        self.criterion_unweighted = nn.CrossEntropyLoss()  # Used for clean validation metrics

        clear_memory()

    def setup_optimizer(self):
        self.optimizer = torch.optim.AdamW(
            self.model.parameters(),
            lr=1e-4,
            weight_decay=1e-4
        )

        self.scheduler = torch.optim.lr_scheduler.StepLR(
            self.optimizer, step_size=10, gamma=0.5
        )

    def train_epoch(self):
        self.model.train()
        total_loss = 0
        total_correct = 0
        total_samples = 0

        progress_bar = tqdm(self.train_loader, desc="Training")

        for batch_idx, (images, labels, _) in enumerate(progress_bar):
            images, labels = images.to(self.device, non_blocking=True), labels.to(self.device, non_blocking=True)

            self.optimizer.zero_grad()
            outputs = self.model(images)
            loss = self.criterion(outputs, labels)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=0.5)
            self.optimizer.step()

            total_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)

            progress_bar.set_postfix({
                'Loss': f'{loss.item():.4f}',
                'Acc': f'{total_correct/total_samples:.4f}'
            })

            if batch_idx % 20 == 0:
                force_clear_memory()

            del images, labels, outputs, loss

        epoch_loss = total_loss / len(self.train_loader)
        epoch_acc = total_correct / total_samples

        force_clear_memory()
        return epoch_loss, epoch_acc

    def validate_epoch(self):
        self.model.eval()
        total_loss = 0
        total_correct = 0
        total_samples = 0
        all_predictions = []
        all_labels = []

        with torch.no_grad():
            for batch_idx, (images, labels, _) in enumerate(tqdm(self.val_loader, desc="Validation")):
                images, labels = images.to(self.device, non_blocking=True), labels.to(self.device, non_blocking=True)

                outputs = self.model(images)
                loss = self.criterion_unweighted(outputs, labels)  # Use unweighted for fair eval

                total_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                total_correct += (predicted == labels).sum().item()
                total_samples += labels.size(0)

                all_predictions.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

                del images, labels, outputs, loss, predicted

                if batch_idx % 10 == 0:
                    clear_memory()

        epoch_loss = total_loss / len(self.val_loader)
        epoch_acc = total_correct / total_samples

        force_clear_memory()
        return epoch_loss, epoch_acc, all_predictions, all_labels

    def generate_vit_visualizations(self, num_samples=3):
        """Generate ViT-specific XAI visualizations"""
        print("\n" + "="*60)
        print("GENERATING VIT-SPECIFIC XAI VISUALIZATIONS")
        print("="*60)

        self.model.eval()

        # Initialize XAI tools
        attention_rollout = AttentionRollout(self.model)
        layer_analysis = TransformerAttributionAnalysis(self.model)
        relevance_gen = RelevanceMapGenerator(self.model)

        sample_count = 0
        for batch_idx, (images, labels, paths) in enumerate(self.val_loader):
            if sample_count >= num_samples:
                break

            for i in range(min(2, len(images))):
                if sample_count >= num_samples:
                    break

                image = images[i:i+1].to(self.device)
                label = labels[i].item()
                path = paths[i]

                # Load original image
                with Image.open(path) as img:
                    orig_img = img.convert('RGB').copy()
                orig_img_resized = orig_img.resize((224, 224))
                orig_img_array = np.array(orig_img_resized)

                class_name = list(self.label_map.keys())[list(self.label_map.values()).index(label)]

                print(f"\nGenerating ViT-XAI for sample {sample_count + 1} - Class: {class_name}")

                try:
                    # 1. Attention Rollout
                    print("  - Attention Rollout...")
                    save_path = os.path.join(output_dir, f"attention_rollout_sample_{sample_count + 1}.png")
                    attention_rollout.visualize(image, orig_img_array, save_path=save_path)

                    # 2. Multi-layer Attention Analysis
                    print("  - Layer Attention Analysis...")
                    save_path = os.path.join(output_dir, f"layer_attention_sample_{sample_count + 1}.png")
                    layer_analysis.visualize_layer_attention(image, orig_img_array, save_path=save_path)

                    # 3. Relevance Map
                    print("  - Relevance Map...")
                    save_path = os.path.join(output_dir, f"relevance_map_sample_{sample_count + 1}.png")
                    relevance_gen.visualize(image, orig_img_array, save_path=save_path)

                except Exception as e:
                    print(f"  Error generating visualizations: {e}")

                sample_count += 1
                force_clear_memory()

        print(f"\nViT-XAI visualizations saved to '{output_dir}' directory")

    def train(self, epochs=15, early_stopping_patience=5):
        patience_counter = 0

        for epoch in range(epochs):
            print(f"\nEpoch {epoch + 1}/{epochs}")
            print("-" * 50)

            train_loss, train_acc = self.train_epoch()
            self.train_losses.append(train_loss)
            self.train_accuracies.append(train_acc)

            val_loss, val_acc, val_preds, val_labels = self.validate_epoch()
            self.val_losses.append(val_loss)
            self.val_accuracies.append(val_acc)

            self.scheduler.step()

            print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
            print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

            f1 = f1_score(val_labels, val_preds, average='weighted')
            precision = precision_score(val_labels, val_preds, average='weighted')
            recall = recall_score(val_labels, val_preds, average='weighted')

            print(f"F1: {f1:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}")

            if val_acc > self.best_val_acc:
                self.best_val_acc = val_acc
                self.best_model_state = self.model.state_dict().copy()
                patience_counter = 0
                print(f"✓ New best model! Val Acc: {val_acc:.4f}")
            else:
                patience_counter += 1

            if patience_counter >= early_stopping_patience:
                print(f"Early stopping at epoch {epoch + 1}")
                break

            if (epoch + 1) % 5 == 0:
                print("\nClassification Report:")
                print(classification_report(val_labels, val_preds,
                                          target_names=list(self.label_map.keys()),
                                          digits=4))
                self.plot_confusion_matrix(val_labels, val_preds, epoch + 1)

            force_clear_memory()

        if self.best_model_state:
            self.model.load_state_dict(self.best_model_state)
            print(f"\nBest model loaded - Val Acc: {self.best_val_acc:.4f}")

        # Generate ViT-XAI visualizations
        self.generate_vit_visualizations(num_samples=3)

    def plot_confusion_matrix(self, true_labels, predictions, epoch):
        cm = confusion_matrix(true_labels, predictions)
        plt.figure(figsize=(10, 8))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                   xticklabels=list(self.label_map.keys()),
                   yticklabels=list(self.label_map.keys()))
        plt.title(f'Confusion Matrix (Epoch {epoch})')
        plt.xlabel('Predicted')
        plt.ylabel('True')
        plt.xticks(rotation=45)
        plt.yticks(rotation=0)
        plt.tight_layout()

        save_path = os.path.join(output_dir, f"confusion_matrix_epoch_{epoch}.png")
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.show()
        plt.close()
        clear_memory()

    def plot_training_history(self):
        epochs = range(1, len(self.train_losses) + 1)

        plt.figure(figsize=(12, 4))

        plt.subplot(1, 2, 1)
        plt.plot(epochs, self.train_losses, 'b-', label='Training', linewidth=2)
        plt.plot(epochs, self.val_losses, 'r-', label='Validation', linewidth=2)
        plt.title('Loss')
        plt.xlabel('Epochs')
        plt.ylabel('Loss')
        plt.legend()
        plt.grid(True, alpha=0.3)

        plt.subplot(1, 2, 2)
        plt.plot(epochs, self.train_accuracies, 'b-', label='Training', linewidth=2)
        plt.plot(epochs, self.val_accuracies, 'r-', label='Validation', linewidth=2)
        plt.title('Accuracy')
        plt.xlabel('Epochs')
        plt.ylabel('Accuracy')
        plt.legend()
        plt.grid(True, alpha=0.3)

        plt.tight_layout()

        save_path = os.path.join(output_dir, "training_history.png")
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.show()
        plt.close()
        clear_memory()

#  STEP 7: Helper Functions
def plot_class_distribution(df, label_map, save_path=None):
    plt.figure(figsize=(10, 6))

    class_counts = df['dx'].value_counts()
    colors = plt.cm.Set3(np.linspace(0, 1, len(class_counts)))

    bars = plt.bar(class_counts.index, class_counts.values, color=colors)
    plt.title('Class Distribution', fontsize=14, pad=20)
    plt.xlabel('Skin Lesion Types')
    plt.ylabel('Number of Images')
    plt.xticks(rotation=45, ha='right')

    for bar, count in zip(bars, class_counts.values):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                f'{count}', ha='center', va='bottom', fontweight='bold')

    plt.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')

    plt.show()
    plt.close()
    clear_memory()

def create_sample_grid(df, label_map, num_samples_per_class=2, save_path=None):
    classes = list(label_map.keys())
    num_classes = len(classes)

    fig, axes = plt.subplots(num_classes, num_samples_per_class,
                           figsize=(3*num_samples_per_class, 2*num_classes))

    if num_classes == 1:
        axes = axes.reshape(1, -1)
    if num_samples_per_class == 1:
        axes = axes.reshape(-1, 1)

    for i, class_name in enumerate(classes):
        class_samples = df[df['dx'] == class_name].sample(
            n=min(num_samples_per_class, len(df[df['dx'] == class_name])))

        for j, (_, row) in enumerate(class_samples.iterrows()):
            if j >= num_samples_per_class:
                break

            try:
                with Image.open(row['path']) as img:
                    img_resized = img.convert('RGB').resize((150, 150))

                axes[i, j].imshow(img_resized)
                axes[i, j].set_title(f'{class_name}', fontsize=8)
                axes[i, j].axis('off')
            except:
                axes[i, j].axis('off')

        for j in range(len(class_samples), num_samples_per_class):
            axes[i, j].axis('off')

    plt.suptitle('Sample Images', fontsize=14)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')

    plt.show()
    plt.close()
    clear_memory()

#  STEP 8: Main Function
def main():
    # Configuration
    IMG_SIZE = 224
    BATCH_SIZE = 8
    NUM_WORKERS = 2
    EPOCHS = 15

    # Update this path to your dataset location
    BASE_PATH = "/kaggle/input/skin-cancer-mnist10000-ham-augmented-dataset"

    print("="*60)
    print("SWIN TRANSFORMER WITH VIT-SPECIFIC XAI TOOLS")
    print("="*60)
    print(f"Configuration: IMG_SIZE={IMG_SIZE}, BATCH_SIZE={BATCH_SIZE}")
    print(f"XAI Methods: Attention Rollout, Layer Analysis, Relevance Maps")

    # Load data
    print("\nLoading data from new directory structure...")
    df, label_map = load_and_preprocess_data(BASE_PATH)

    # Visualizations
    plot_class_distribution(df, label_map,
                           save_path=os.path.join(output_dir, 'class_distribution.png'))

    create_sample_grid(df, label_map, num_samples_per_class=2,
                      save_path=os.path.join(output_dir, 'sample_grid.png'))

    # Split data
    train_df = df[df['split'] == 'train'].copy()
    val_df = df[df['split'] == 'val'].copy()

    print(f"\nDataset Split:")
    print(f"Training: {len(train_df)}")
    print(f"Validation: {len(val_df)}")

    # Transforms
    train_transform = get_transforms(IMG_SIZE, mode='train')
    val_transform = get_transforms(IMG_SIZE, mode='val')

    # Datasets
    train_dataset = HAM10000Dataset(train_df, transform=train_transform)
    val_dataset = HAM10000Dataset(val_df, transform=val_transform)

    # Samplers
    train_labels = train_df['label'].values
    class_counts = np.bincount(train_labels)
    # Calculate weights for sampler based on inverse frequency
    class_weights_sampler = 1.0 / class_counts
    sample_weights = class_weights_sampler[train_labels]
    # Use the same enhanced weights for the sampler for consistency
    # We'll re-use the logic from setup_loss_function for the sampler weights
    # Since we want the sampler to reflect the same bias correction as the loss function.
    # In practice, you might want to use the *same* set of weights for both.
    # For simplicity and to ensure consistency, let's calculate the final weights here too.
    # This is a bit redundant but ensures perfect alignment.
    class_weights_final = compute_class_weight(
        'balanced',
        classes=np.unique(train_labels),
        y=train_labels
    )
    # Apply the same boosts as in the loss function
    mel_class_index = list(label_map.keys()).index('mel')
    bkl_class_index = list(label_map.keys()).index('bkl')
    avg_top_accuracy = (0.972 + 0.996 + 0.983 + 0.970 + 1.000) / 5
    mel_accuracy = 0.890
    bkl_accuracy = 0.918
    mel_boost_factor = (avg_top_accuracy / mel_accuracy) ** 1.5
    bkl_boost_factor = (avg_top_accuracy / bkl_accuracy) ** 1.5
    class_weights_final[mel_class_index] *= mel_boost_factor
    class_weights_final[bkl_class_index] *= bkl_boost_factor
    # Now convert to sample weights for the sampler
    sample_weights_for_sampler = class_weights_final[train_labels]
    sampler = WeightedRandomSampler(sample_weights_for_sampler, len(sample_weights_for_sampler))

    # Create data loaders with drop_last=True to avoid batch size of 1
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        sampler=sampler,
        num_workers=NUM_WORKERS,
        pin_memory=False,
        persistent_workers=False,
        drop_last=True  # Drop last incomplete batch to avoid BatchNorm error
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=False,
        persistent_workers=False,
        drop_last=False  # Keep all validation samples
    )

    force_clear_memory()

    # Model
    print("\nCreating Swin Transformer model...")
    model = EnhancedSwinTransformer(
        num_classes=len(label_map),
        dropout_rate=0.2,
        pretrained=True
    ).to(device)

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")

    force_clear_memory()

    # Trainer
    trainer = EnhancedTrainer(model, train_loader, val_loader,
                            len(label_map), label_map, device)

    # Train
    print("\nStarting training...")
    trainer.train(epochs=EPOCHS, early_stopping_patience=5)

    # Training history
    trainer.plot_training_history()

    # Save model
    model_save_path = 'swin_vit_xai_model.pth'
    torch.save({
        'model_state_dict': model.state_dict(),
        'label_map': label_map,
        'model_config': {
            'num_classes': len(label_map),
            'img_size': IMG_SIZE,
            'dropout_rate': 0.2
        },
        'training_history': {
            'train_losses': trainer.train_losses,
            'val_losses': trainer.val_losses,
            'train_accuracies': trainer.train_accuracies,
            'val_accuracies': trainer.val_accuracies
        }
    }, model_save_path)

    print(f"\n{'='*60}")
    print("TRAINING COMPLETED!")
    print(f"{'='*60}")
    print(f"Model saved: {model_save_path}")
    print(f"Best validation accuracy: {trainer.best_val_acc:.4f}")
    print(f"Visualizations saved to: {os.path.abspath(output_dir)}")

    print("\nGenerated ViT-XAI Visualizations:")
    print("  ✓ Attention Rollout - Shows aggregated attention flow")
    print("  ✓ Layer Attention Analysis - Multi-layer feature visualization")
    print("  ✓ Relevance Maps - Gradient-based attribution")

    force_clear_memory()

if __name__ == "__main__":
    os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

    try:
        main()
    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()
        force_clear_memory()